# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/uzairrateef-rgb/my-flyrank-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
!git clone https://github.com/uzairrateef-rgb/my-flyrank-internship.git
%cd my-flyrank-internship

%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"
print("Connected. Ready to query.")


Cloning into 'my-flyrank-internship'...
remote: Enumerating objects: 133, done.
remote: Counting objects: 100% (133/133), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 133 (delta 41), reused 96 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (133/133), 1.88 MiB | 11.58 MiB/s, done.
Resolving deltas: 100% (41/41), done.
/content/my-flyrank-internship/my-flyrank-internship
Connected. Ready to query.


In [11]:
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{WAREHOUSE}/dim_content.parquet') LIMIT 1").show()

┌────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│        column_name         │ column_type │  null   │   key   │ default │  extra  │
│          varchar           │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ client_hash_id             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_hash_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ url_hash_id                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_char_count         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_token_count        │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ url_char_count             │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ content_created_date       │ DATE        │ YES     │ NULL    │ 

In [12]:
q = f"""
SELECT
  CASE
    WHEN gsc_avg_position <= 3 THEN 'top_3'
    WHEN gsc_avg_position <= 10 THEN 'page_1'
    WHEN gsc_avg_position <= 20 THEN 'page_2'
    ELSE 'deep'
  END AS position_bucket,
  COUNT(*) AS n,
  SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS ctr
FROM read_parquet('{WAREHOUSE}/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE gsc_data_available IS TRUE AND gsc_impressions > 0
GROUP BY 1
ORDER BY ctr DESC
"""
con.sql(q).show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬─────────┬───────────────────────┐
│ position_bucket │    n    │          ctr          │
│     varchar     │  int64  │        double         │
├─────────────────┼─────────┼───────────────────────┤
│ top_3           │  727362 │ 0.0038027456350242985 │
│ page_1          │ 1456122 │ 0.0032346197089746275 │
│ page_2          │  519223 │ 0.0031460212728466742 │
│ deep            │  908354 │  0.001314496204492777 │
└─────────────────┴─────────┴───────────────────────┘



In [13]:
q = f"""
SELECT
  CASE
    WHEN DATE_DIFF('day', c.content_updated_date, f.report_date) >= 180 THEN 'stale_180plus'
    WHEN DATE_DIFF('day', c.content_updated_date, f.report_date) >= 90  THEN 'stale_90_179'
    WHEN DATE_DIFF('day', c.content_updated_date, f.report_date) >= 30  THEN 'recent_30_89'
    ELSE 'fresh_under_30'
  END AS staleness_bucket,
  COUNT(*) AS n,
  SUM(f.gsc_clicks) * 1.0 / NULLIF(SUM(f.gsc_impressions), 0) AS ctr
FROM read_parquet('{WAREHOUSE}/fact_content_daily_performance/month=2026-03/*.parquet') f
JOIN read_parquet('{WAREHOUSE}/dim_content.parquet') c
  ON f.content_hash_id = c.content_hash_id
WHERE f.gsc_data_available IS TRUE AND f.gsc_impressions > 0
  AND c.content_updated_date IS NOT NULL
GROUP BY 1
ORDER BY 1
"""
con.sql(q).show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────┬─────────┬───────────────────────┐
│ staleness_bucket │    n    │          ctr          │
│     varchar      │  int64  │        double         │
├──────────────────┼─────────┼───────────────────────┤
│ fresh_under_30   │ 3495262 │  0.002946905702400531 │
│ recent_30_89     │  105224 │ 0.0020265285449093296 │
│ stale_180plus    │    1412 │ 0.0015859962406015037 │
│ stale_90_179     │    9163 │  0.002735727076948388 │
└──────────────────┴─────────┴───────────────────────┘



## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
"Signal check 1: CTR vs. position (behind FlyRank's CTR-fix flag): CONFIRMED"
"CTR falls cleanly as position worsens, top_3 (0.380%) → page_1 (0.323%) → page_2 (0.315%) → deep (0.131%), monotonic across 3.6M rows. This is the strongest, most reliable signal available."

"Signal check 2: staleness vs. CTR (behind FlyRank's refresh flag): MIXED"
"Freshest content (under 30 days) has the highest CTR (0.295%) and the most-stale (180+ days) has the lowest (0.159%) — directionally supportive. But the middle buckets don't follow a clean order (`recent_30_89` at 0.203% is lower than `stale_90_179` at 0.274%), and the stale buckets have very small n (1,412 and 9,163) next to 3.5M fresh rows. I don't trust this signal enough to weight it heavily in the rule — a clearly negative/mixed result here is useful: it stops me from overbuilding on a weak assumption."

"The rule: Because Signal 1 is confirmed and Signal 2 is not, my rule leans primarily on position-based CTR underperformance, using staleness only as a light secondary tiebreaker, not a primary driver."

"Score = (expected CTR for the page's position bucket − actual CTR) × impressions, if positive."
"This flags pages getting *fewer clicks than typical for their ranking position*, weighted by how much traffic they'd gain if fixed."

"Reason codes:"
"- `CTR_GAP` — actual CTR is below the position-bucket average (the core signal)"
"- `LOW_VOLUME` — insufficient impressions to trust the CTR calculation (excluded from ranking, not scored)"

"Action label: `review_ctr` (score > 0) or `monitor` (score ≤ 0 or excluded)"


'Action label: `review_ctr` (score > 0) or `monitor` (score ≤ 0 or excluded)'

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Aggregate to content level for March
content_agg = con.sql(f"""
    SELECT
      content_hash_id,
      SUM(gsc_impressions) AS impressions,
      SUM(gsc_clicks) AS clicks,
      AVG(gsc_avg_position) AS avg_position,
      SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS ctr
    FROM read_parquet('{WAREHOUSE}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
    HAVING SUM(gsc_impressions) > 0
""").df()

import numpy as np

def position_bucket(p):
    if p <= 3: return 'top_3'
    elif p <= 10: return 'page_1'
    elif p <= 20: return 'page_2'
    else: return 'deep'

content_agg['position_bucket'] = content_agg['avg_position'].apply(position_bucket)

# Expected CTR per bucket, from Signal 1's confirmed numbers
expected_ctr = {'top_3': 0.003803, 'page_1': 0.003235, 'page_2': 0.003146, 'deep': 0.001314}
content_agg['expected_ctr'] = content_agg['position_bucket'].map(expected_ctr)

MIN_IMPRESSIONS = 100  # avoid scoring noisy low-volume pages
content_agg['low_volume'] = content_agg['impressions'] < MIN_IMPRESSIONS

content_agg['ctr_gap'] = content_agg['expected_ctr'] - content_agg['ctr']
content_agg['score'] = np.where(
    content_agg['low_volume'], 0,
    (content_agg['ctr_gap'] * content_agg['impressions']).clip(lower=0)
)

content_agg['reason_code'] = np.where(content_agg['low_volume'], 'LOW_VOLUME', 'CTR_GAP')
content_agg['action'] = np.where(content_agg['score'] > 0, 'review_ctr', 'monitor')

queue = content_agg.sort_values('score', ascending=False)

import os
os.makedirs('work/outputs', exist_ok=True)
queue.to_csv('work/outputs/baseline_action_score.csv', index=False)
print(f"Wrote {len(queue)} rows to work/outputs/baseline_action_score.csv")
queue.head(20)


Wrote 176738 rows to work/outputs/baseline_action_score.csv


,content_hash_id,impressions,clicks,avg_position,ctr,position_bucket,expected_ctr,low_volume,ctr_gap,score,reason_code,action
106898,content_44f34c0a90047651,212404.0,24.0,7.346909,0.000113,page_1,0.003235,False,0.003122,663.126940,CTR_GAP,review_ctr
19411,content_8d7d99f109e19aa2,203497.0,289.0,2.563756,0.001420,top_3,0.003803,False,0.002383,484.899091,CTR_GAP,review_ctr
273,content_8e1334d6356668e3,134984.0,1.0,4.545582,0.000007,page_1,0.003235,False,0.003228,435.673240,CTR_GAP,review_ctr
7627,content_34a70fea29d15f24,143019.0,43.0,3.219473,0.000301,page_1,0.003235,False,0.002934,419.666465,CTR_GAP,review_ctr
88561,content_fec55986a1868d62,124075.0,1.0,9.385150,0.000008,page_1,0.003235,False,0.003227,400.382625,CTR_GAP,review_ctr
22759,content_7c6373141eae744a,132593.0,83.0,5.789019,0.000626,page_1,0.003235,False,0.002609,345.938355,CTR_GAP,review_ctr
120358,content_f6116743b00afc2d,107584.0,15.0,9.536301,0.000139,page_1,0.003235,False,0.003096,333.034240,CTR_GAP,review_ctr
22788,content_acbcc847f8996314,170808.0,262.0,3.361195,0.001534,page_1,0.003235,False,0.001701,290.563880,CTR_GAP,review_ctr
56606,content_cd3d932d4e1c8db0,89332.0,4.0,7.786219,0.000045,page_1,0.003235,False,0.003190,284.989020,CTR_GAP,review_ctr
107859,content_306bc78dff1eb683,80821.0,35.0,1.488604,0.000433,top_3,0.003803,False,0.003370,272.362263,CTR_GAP,review_ctr


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
queue.head(20)[['content_hash_id', 'impressions', 'clicks', 'position_bucket', 'ctr', 'score', 'reason_code', 'action']]


,content_hash_id,impressions,clicks,position_bucket,ctr,score,reason_code,action
106898,content_44f34c0a90047651,212404.0,24.0,page_1,0.000113,663.126940,CTR_GAP,review_ctr
19411,content_8d7d99f109e19aa2,203497.0,289.0,top_3,0.001420,484.899091,CTR_GAP,review_ctr
273,content_8e1334d6356668e3,134984.0,1.0,page_1,0.000007,435.673240,CTR_GAP,review_ctr
7627,content_34a70fea29d15f24,143019.0,43.0,page_1,0.000301,419.666465,CTR_GAP,review_ctr
88561,content_fec55986a1868d62,124075.0,1.0,page_1,0.000008,400.382625,CTR_GAP,review_ctr
22759,content_7c6373141eae744a,132593.0,83.0,page_1,0.000626,345.938355,CTR_GAP,review_ctr
120358,content_f6116743b00afc2d,107584.0,15.0,page_1,0.000139,333.034240,CTR_GAP,review_ctr
22788,content_acbcc847f8996314,170808.0,262.0,page_1,0.001534,290.563880,CTR_GAP,review_ctr
56606,content_cd3d932d4e1c8db0,89332.0,4.0,page_1,0.000045,284.989020,CTR_GAP,review_ctr
107859,content_306bc78dff1eb683,80821.0,35.0,top_3,0.000433,272.362263,CTR_GAP,review_ctr


1. content_44f34c0a — 212K impressions, only 24 clicks at page_1 wrong if this page's true intent is informational (low clicks are expected, not a CTR problem).

2. content_8d7d99f1 — top_3 position but CTR far below expected wrong if title/meta was recently changed and hasn't re-indexed yet.

3. content_8e1334d6 — 135K impressions, 1 click wrong if this is a single outlier day skewing the month average, not a real pattern.

4. content_34a70fea — page_1, CTR 10x below expected wrong if the snippet is accurate but the query intent doesn't match the page.

5. content_fec55986 — 124K impressions, 1 click same single-click fragility as #3; needs a sanity check on daily distribution.

6. content_7c637314 — page_1, moderate clicks but still a real gap most trustworthy pick so far given decent click volume (83).

7. content_f6116743 — 15 clicks on 107K impressions wrong if ranking position is unstable day-to-day (avg masks volatility).

8. content_acbcc847 — 262 clicks, largest click count in top 20 strongest, most reliable pick here.

9. content_cd3d932d — 4 clicks on 89K impressions wrong if this is a seasonal/one-off spike in impressions, not sustained visibility.

10. content_306bc78d — top_3 with real gap wrong if competitors also occupy top_3 and CTR is split three ways structurally.

11. content_b99ea686 — 361 clicks, second-highest click count reliable, worth prioritizing.

12. content_046fc480 — 6 clicks, low signal wrong if content type structurally gets lower CTR (e.g. reference page) regardless of ranking.

13. content_9c057b66 — page_2, 1 click weakest data support in top 20; flag for exclusion, not action.

14. content_f43118e0 — 191 clicks, solid volume trustworthy pick.

15. content_9540d884 — 11 clicks wrong if position is borderline page_1/page_2 and misbucketed.

16. content_9ef3d751 — top_3, 92 clicks reasonably supported, worth reviewing.

17. content_425715547c — 3 clicks same single-digit-click fragility as others; low confidence.

18. content_c46df0fa — top_3, 42 clicks moderate confidence.

19. content_36fc1ee5 — 16 clicks wrong if impressions are inflated by bot/scraper traffic rather than real users.

20. content_e578ac84 — 163 clicks solid, trustworthy pick to close out the top 20.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
"Weak picks: Rows #3, #5, #9, #13, #17 all have 1–4 clicks against huge impression counts (83K–212K). A CTR built on 1–4 clicks is statistically unstable one extra click doubles or triples the rate. These shouldn't be trusted at the same confidence as picks like #8 (262 clicks) or #11 (361 clicks). A future version of this rule should require a minimum click count, not just a minimum impression count, before scoring."
"Leakage check: The rule uses only `gsc_impressions`, `gsc_clicks`, and `gsc_avg_position` all observed within the same March window being scored, with no future month, no product-generated flags (e.g. no `is_published`/`is_deleted` from `dim_content`), and no label-derived column. `expected_ctr` come from Signal 1's aggregate bucket averages (computed on the same data, not a hidden future outcome), so this is a legitimate baseline rule, not a leaked one."

used_columns = {'content_hash_id', 'impressions', 'clicks', 'avg_position', 'ctr',
                 'position_bucket', 'expected_ctr'}
excluded_from_dim_content = {'is_published', 'is_deleted', 'last_optimized_date',
                              'optimization_eligible_date', 'provider_used', 'model_used'}
print("Used:", used_columns)
print("Deliberately excluded (product flags / future-decision fields):", excluded_from_dim_content)


Used: {'expected_ctr', 'impressions', 'avg_position', 'position_bucket', 'ctr', 'content_hash_id', 'clicks'}
Deliberately excluded (product flags / future-decision fields): {'optimization_eligible_date', 'is_deleted', 'model_used', 'is_published', 'provider_used', 'last_optimized_date'}


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.